In [32]:
"""
HERA Atmospheric Sensor Data — Analysis Script
================================================
Reads "HERA Data.xlsx" (sheets "850am to1250pm" and "450pm to 850pm"),
cleans it, detects the separate logger runs hiding inside the second
sheet, and builds "HERAAnalysis.xlsx" containing:
  - Overview         : summary
  - S1_Data/S2_Data  : cleaned readings, tagged by run (s2 had 3 separate runs)
  - Segment_Stats    : min/mean/max/std per sensor per run (live formulas)
  - Trends           : minute-binned line charts per run
  - Correlations     : correlation-with-temperature matrix per run

Setup
-----
    Ensure you have installed pandas and openpyxl in environment

Usage
-----
    python hera_sensor_analysis.py
(edit SRC / OUT below if your filenames or paths differ)
"""

import math
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.chart import LineChart, Reference
from openpyxl.utils import get_column_letter
from openpyxl.chart.layout import Layout, ManualLayout


In [33]:
# Config — edit these two paths for your machine
SRC = "HERA Data.xlsx" # The file you want to analyze
OUT = "HERAAnalysis.xlsx" # The file you want created by the script

# Change if more or less sheets
SHEET_1 = "850am to1250pm" 
SHEET_2 = "450pm to 850pm"

# Styling constants
FONT = "Arial"
HEADER_FILL = PatternFill("solid", fgColor="1F4E78")
HEADER_FONT = Font(name=FONT, bold=True, color="FFFFFF", size=11)
TITLE_FONT = Font(name=FONT, bold=True, size=14, color="1F4E78")
SUB_FONT = Font(name=FONT, bold=True, size=11, color="1F4E78")
BODY_FONT = Font(name=FONT, size=10)
THIN = Side(style="thin", color="CCCCCC")
BORDER = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)

COLS = ['Time_Stamp_s', 'HTU_Temp_C', 'HTU_Humidity_Percent', 'UV_Raw', 'DPS_Temp_C', 'DPS_Pressure_hPa', 'PM_0.3um', 'PM_0.5um', 'PM_1.0um', 'PM_2.5um', 'PM_5.0um', 'PM_10um']

METRICS = ['HTU_Temp_C', 'HTU_Humidity_Percent', 'UV_Raw', 'DPS_Temp_C', 'DPS_Pressure_hPa', 'PM_0.3um', 'PM_0.5um', 'PM_1.0um', 'PM_2.5um', 'PM_5.0um', 'PM_10um']


In [34]:
# Step 1: load & split each sheet into its separate logger runs
def split_into_runs(df: pd.DataFrame) -> list[pd.DataFrame]:
    """A sheet can contain multiple logger runs concatenated together,
    each restarting Time_Stamp_s at 0. Detect this via repeated header
    rows mid-sheet and split into separate cleaned DataFrames."""
    df = df.dropna(axis=1, how='all')
    is_header_row = df['Time_Stamp_s'].astype(str) == 'Time_Stamp_s'
    bounds = [0] + [i + 1 for i in df.index[is_header_row]] + [len(df)]

    runs = []
    for i in range(len(bounds) - 1):
        seg = df.iloc[bounds[i]:bounds[i + 1]].copy()
        seg = seg[seg['Time_Stamp_s'].astype(str) != 'Time_Stamp_s']
        for c in seg.columns:
            seg[c] = pd.to_numeric(seg[c], errors='coerce')
        seg = seg.dropna(subset=['Time_Stamp_s']).reset_index(drop=True)
        runs.append(seg)
    return runs

def load_data(src_path: str):
    s1_raw = pd.read_excel(src_path, sheet_name=SHEET_1, header=0)
    s2_raw = pd.read_excel(src_path, sheet_name=SHEET_2, header=0)

    s1_runs = split_into_runs(s1_raw)
    s2_runs = split_into_runs(s2_raw)

    # Adjust this mapping if your file has a different number of runs per sheet
    # print(len(s1_runs)), print(len(s2_runs)) to check first.
    segments = {'S1': s1_runs[0]}
    labels = ['S2_A', 'S2_B', 'S2_C']
    for label, run in zip(labels, s2_runs):
        segments[label] = run
    return segments


In [35]:
# Step 2: analysis helpers
def describe_segment(label: str, df: pd.DataFrame) -> dict:
    return {
        'label': label,
        'rows': len(df),
        'duration_s': df['Time_Stamp_s'].max(),
        'temp_start': df['HTU_Temp_C'].iloc[0],
        'temp_end': df['HTU_Temp_C'].iloc[-1],
        'pct_negative_humidity': round((df['HTU_Humidity_Percent'] < 0).mean() * 100, 1),
        'pm03_mean': round(df['PM_0.3um'].mean(), 1),
        'pm03_max': df['PM_0.3um'].max(),
        'corr_temp_uv': round(df[['DPS_Temp_C', 'UV_Raw']].corr().iloc[0, 1], 2),
        'corr_temp_humidity': round(df[['DPS_Temp_C', 'HTU_Humidity_Percent']].corr().iloc[0, 1], 2),
        'corr_temp_pm03': round(df[['DPS_Temp_C', 'PM_0.3um']].corr().iloc[0, 1], 2),
    }



In [36]:
# Step 3: workbook builders
def nice_axis(data_max, n_div=6):
    """Clean axis max + majorUnit spanning 0...max in exactly n_div equal steps.
    Using the SAME n_div on both axes makes their gridlines coincide, so a
    dual-axis chart shows one shared set of gridlines instead of two."""
    if data_max is None or data_max <= 0:
        return float(n_div), 1.0
    raw_step = data_max / n_div
    mag = 10 ** math.floor(math.log10(raw_step))
    for m in (1, 2, 2.5, 5, 10):
        step = m * mag
        if step * n_div >= data_max:
            return step * n_div, step
    step = 10 * mag
    return step * n_div, step

def style_header_row(ws, row, ncols, start_col=1):
    for c in range(start_col, start_col + ncols):
        cell = ws.cell(row=row, column=c)
        cell.font = HEADER_FONT
        cell.fill = HEADER_FILL
        cell.alignment = Alignment(horizontal="center", vertical="center")
        cell.border = BORDER

def style_time_axis(chart, skip=10):
    chart.x_axis.delete = False
    chart.x_axis.tickLblPos = "low"
    chart.x_axis.tickLblSkip = skip
    chart.x_axis.tickMarkSkip = skip
    chart.x_axis.number_format = "0"
    chart.x_axis.majorTickMark = "out"
    if chart.x_axis.title is not None:
        chart.x_axis.title.overlay = False
        chart.x_axis.title.layout = Layout(
            manualLayout=ManualLayout(xMode='edge', yMode='edge', x=0.40, y=0.93)
        )

def style_value_axis(chart):
    chart.y_axis.delete = False
    chart.y_axis.tickLblPos = "nextTo"
    chart.y_axis.majorTickMark = "out"
    chart.y_axis.number_format = "0"
    if chart.y_axis.title is not None:
        chart.y_axis.title.overlay = False
        chart.y_axis.title.layout = Layout(
            manualLayout=ManualLayout(xMode='edge', yMode='edge', x=0.01, y=0.30)
        )

def style_chart_title(chart):
    if chart.title is not None:
        chart.title.overlay = False

def style_legend(chart, x=0.82, y=0.12, w=0.18, h=0.25):
    chart.legend.overlay = False
    chart.legend.layout = Layout(
        manualLayout=ManualLayout(xMode='edge', yMode='edge', x=x, y=y, w=w, h=h)
    )

def autosize(ws, ncols, start_col=1, width=14):
    for c in range(start_col, start_col + ncols):
        ws.column_dimensions[get_column_letter(c)].width = width

def build_overview_sheet(wb, seg_info: dict):
    ov = wb.create_sheet("Overview")
    ov.sheet_view.showGridLines = False
    ov['B2'] = "HERA Analysis Summary"
    ov['B2'].font = TITLE_FONT
    ov['B3'] = ("Source: " + SRC + "  |  Sensors: HTU21D (temp/humidity), analog UV, "
                "DPS310 (temp/pressure), PM optical particle counter (0.3-10um)")
    ov['B3'].font = BODY_FONT

    r = 5
    ov.cell(row=r, column=2, value="1. Data Structure").font = SUB_FONT
    r += 1
    notes = [
        f"Sheet \"{SHEET_1}\" contains 1 continuous logger run.",
        f"Sheet \"{SHEET_2}\" contains {len(seg_info) - 1} separate logger runs concatenated "
        "together — each one restarts Time_Stamp_s at 0 (detected via repeated header rows "
        "mid-sheet). Treating the sheet as one continuous timeline would mix different "
        "regimes together.",
    ]
    for label, info in seg_info.items():
        notes.append(
            f"   - {label}: {info['rows']:,} rows, ~{info['duration_s'] / 3600:.1f} hours, "
            f"temp {info['temp_start']}C -> {info['temp_end']}C."
        )
    for n in notes:
        ov.cell(row=r, column=2, value=n).font = BODY_FONT
        ov.cell(row=r, column=2).alignment = Alignment(wrap_text=True, vertical="top")
        r += 1

    r += 1
    ov.cell(row=r, column=2, value="2. Key Findings (auto-generated from the data below)").font = SUB_FONT
    r += 1
    findings = []
    neg_runs = [l for l, i in seg_info.items() if i['pct_negative_humidity'] > 0]
    pos_runs = [l for l, i in seg_info.items() if i['pct_negative_humidity'] == 0]
    findings.append(
        f"Humidity sensor anomaly: {', '.join(neg_runs)} show negative (physically invalid) "
        f"humidity readings at points in the run, while {', '.join(pos_runs) or 'none'} never do. "
    )
    hottest = max(seg_info.values(), key=lambda i: i['corr_temp_uv'])
    findings.append(
        f"Temperature-UV correlation is strongest in {hottest['label']} (r={hottest['corr_temp_uv']})"
    )
    pm_runs = sorted(seg_info.values(), key=lambda i: -i['pm03_mean'])
    top_pm = pm_runs[0]
    findings.append(
        f"Highest particulate load is in {top_pm['label']} (mean PM_0.3um={top_pm['pm03_mean']}, "
        f"max={top_pm['pm03_max']}, correlation with temp={top_pm['corr_temp_pm03']}) "
    )

    for f in findings:
        ov.cell(row=r, column=2, value="• " + f).font = BODY_FONT
        ov.row_dimensions[r].height = 30
        ov.cell(row=r, column=2).alignment = Alignment(wrap_text=True, vertical="top")
        r += 1

    r += 1
    ov.cell(row=r, column=2, value="3. What's in this workbook").font = SUB_FONT
    r += 1
    for n in ["Segment_Stats — min/mean/max/std for every sensor, per run (live formulas).",
              "Trends — time-binned charts of temperature, humidity, UV, and PM for each run.",
              "S1_Data / S2_Data — full cleaned readings (S2_Data tagged by run).",
              "Correlations — correlation matrix per run."]:
        ov.cell(row=r, column=2, value="• " + n).font = BODY_FONT
        r += 1

    ov.column_dimensions['A'].width = 3
    ov.column_dimensions['B'].width = 110

def write_data_sheet(ws, df):
    ws.sheet_view.showGridLines = False
    for j, c in enumerate(COLS, start=1):
        ws.cell(row=1, column=j, value=c)
    style_header_row(ws, 1, len(COLS))
    for i, row in enumerate(df[COLS].itertuples(index=False), start=2):
        for j, val in enumerate(row, start=1):
            ws.cell(row=i, column=j, value=val)
    autosize(ws, len(COLS), width=13)
    ws.freeze_panes = "A2"
    return len(df)

def build_data_sheets(wb, segments: dict):
    s1_ws = wb.create_sheet("S1_Data")
    n_s1 = write_data_sheet(s1_ws, segments['S1'])

    s2_ws = wb.create_sheet("S2_Data")
    s2_ws.sheet_view.showGridLines = False
    s2_ws.cell(row=1, column=1, value="Segment")
    for j, c in enumerate(COLS, start=2):
        s2_ws.cell(row=1, column=j, value=c)
    style_header_row(s2_ws, 1, len(COLS) + 1)

    r = 2
    seg_row_ranges = {}
    for label in segments:
        if label == 'S1':
            continue
        start_r = r
        for row in segments[label][COLS].itertuples(index=False):
            s2_ws.cell(row=r, column=1, value=label)
            for j, val in enumerate(row, start=2):
                s2_ws.cell(row=r, column=j, value=val)
            r += 1
        seg_row_ranges[label] = (start_r, r - 1)
    autosize(s2_ws, len(COLS) + 1, width=13)
    s2_ws.freeze_panes = "A2"
    return n_s1, seg_row_ranges

def col_letter_in(sheetname, metric):
    idx = COLS.index(metric) + (1 if sheetname == "S1_Data" else 2)
    return get_column_letter(idx)

def build_segment_stats_sheet(wb, segments, n_s1, seg_row_ranges):
    seg_refs = {'S1': ("S1_Data", 2, n_s1 + 1)}
    for label, (r1, r2) in seg_row_ranges.items():
        seg_refs[label] = ("S2_Data", r1, r2)

    ws = wb.create_sheet("Segment_Stats")
    ws.sheet_view.showGridLines = False
    ws.cell(row=1, column=1, value="HERA Sensor Summary Statistics by Run").font = TITLE_FONT

    row = 3
    for seg_label, (sheetname, r1, r2) in seg_refs.items():
        ws.cell(row=row, column=1, value=f"Run: {seg_label}").font = SUB_FONT
        row += 1
        headers = ["Sensor", "Min", "Mean", "Max", "Std Dev", "N"]
        for j, h in enumerate(headers, start=1):
            ws.cell(row=row, column=j, value=h)
        style_header_row(ws, row, len(headers))
        row += 1
        for m in METRICS:
            cl = col_letter_in(sheetname, m)
            rng = f"{sheetname}!{cl}{r1}:{cl}{r2}"
            ws.cell(row=row, column=1, value=m).font = BODY_FONT
            ws.cell(row=row, column=2, value=f"=MIN({rng})")
            ws.cell(row=row, column=3, value=f"=AVERAGE({rng})")
            ws.cell(row=row, column=4, value=f"=MAX({rng})")
            ws.cell(row=row, column=5, value=f"=STDEV({rng})")
            ws.cell(row=row, column=6, value=f"=COUNT({rng})")
            for c in range(2, 6):
                ws.cell(row=row, column=c).number_format = "0.00"
            row += 1
        row += 1
    autosize(ws, 6, width=16)
    return seg_refs

def build_trends_sheet(wb, segments):
    ws = wb.create_sheet("Trends")
    ws.sheet_view.showGridLines = False
    ws.cell(row=1, column=1, value="Time-Binned Trends (1-min bins) per Run").font = TITLE_FONT

    pm_sizes = ['PM_0.3um', 'PM_0.5um', 'PM_1.0um', 'PM_2.5um', 'PM_5.0um', 'PM_10um']
    bin_metrics = ['HTU_Temp_C', 'HTU_Humidity_Percent', 'UV_Raw'] + pm_sizes
    labels = list(segments.keys())
    data_start_col = 50
    col_cursor = data_start_col

    for idx, seg_label in enumerate(labels):
        df = segments[seg_label].copy()
        df['bin'] = (df['Time_Stamp_s'] // 60) * 60
        binned = df.groupby('bin')[bin_metrics].mean().reset_index()
        binned['bin'] = binned['bin'] / 60

        start_col = col_cursor
        ws.cell(row=1, column=start_col, value=f"{seg_label} binned data")
        headers = ['Time_min'] + bin_metrics
        for j, h in enumerate(headers, start=start_col):
            ws.cell(row=2, column=j, value=h)
        for i, rowvals in enumerate(binned.itertuples(index=False), start=3):
            for j, val in enumerate(rowvals, start=start_col):
                ws.cell(row=i, column=j, value=val)
        n_bins = len(binned)
        cats = Reference(ws, min_col=start_col, min_row=3, max_row=2 + n_bins)

        anchor_col = get_column_letter(1 + idx * 12)
        anchor_row = 2

        # Chart: Temp & Humidity vs time
        chart_th = LineChart()
        chart_th.title = f"{seg_label}: Temp & Humidity vs time"
        chart_th.y_axis.title = "Temp C / Humid %"
        chart_th.x_axis.title = "Time (min)"
        chart_th.height, chart_th.width = 9.5, 18
        chart_th.legend.position = 'r'
        style_legend(chart_th)

        for metric in ['HTU_Temp_C', 'HTU_Humidity_Percent']:
            data = Reference(ws, min_col=start_col + 1 + bin_metrics.index(metric), min_row=2, max_row=2 + n_bins)
            chart_th.add_data(data, titles_from_data=True)

        chart_th.set_categories(cats)
        style_time_axis(chart_th)
        style_value_axis(chart_th)
        style_chart_title(chart_th)
        ws.add_chart(chart_th, f"{anchor_col}{anchor_row}")
        anchor_row += 26

        # One dual-axis chart per particle size: UV (left) & PM_x (right) vs time
        for pm in pm_sizes:
            pm_max = binned[pm].max()
            pm_fmt = "0.00" if pm_max < 10 else ("0.0" if pm_max < 100 else "0")
            c_uv = LineChart()
            c_uv.title = f"{seg_label}: UV & {pm} vs time"
            c_uv.y_axis.title = "UV raw"
            c_uv.x_axis.title = "Time (min)"
            c_uv.height, c_uv.width = 9.5, 18

            data_uv = Reference(ws, min_col=start_col + 1 + bin_metrics.index('UV_Raw'),
                                 min_row=2, max_row=2 + n_bins)
            c_uv.add_data(data_uv, titles_from_data=True)
            GRID_DIVISIONS = 6
            uv_axis_max, uv_unit = nice_axis(binned['UV_Raw'].max(), GRID_DIVISIONS)
            c_uv.y_axis.scaling.min = 0
            c_uv.y_axis.scaling.max = uv_axis_max
            c_uv.y_axis.majorUnit  = uv_unit
            c_uv.set_categories(cats)
            c_uv.series[0].graphicalProperties.line.solidFill = "4F81BD"  # blue

            style_time_axis(c_uv)
            style_value_axis(c_uv)
            style_chart_title(c_uv)
            c_uv.legend.position = 'r'
            c_uv.legend.overlay = False
            c_uv.legend.layout = Layout(
                manualLayout=ManualLayout(xMode='edge', yMode='edge', x=0.90, y=0.12, w=0.10, h=0.25)
            )
            style_legend(c_uv)

            c_pm = LineChart()
            c_pm.y_axis.axId = 200
            c_pm.y_axis.title = "PM count"
            c_pm.y_axis.title.overlay = False
            c_pm.y_axis.title.layout = Layout(
                manualLayout=ManualLayout(xMode='edge', yMode='edge', x=0.79, y=0.30)
            )
            c_pm.y_axis.crosses = "max"
            c_pm.y_axis.delete = False
            c_pm.y_axis.majorTickMark = "out"
            c_pm.y_axis.number_format = pm_fmt

            data_pm = Reference(ws, min_col=start_col + 1 + bin_metrics.index(pm), min_row=2, max_row=2 + n_bins)
            c_pm.add_data(data_pm, titles_from_data=True)
            pm_axis_max, pm_unit = nice_axis(pm_max, GRID_DIVISIONS)
            c_pm.y_axis.scaling.min = 0
            c_pm.y_axis.scaling.max = pm_axis_max
            c_pm.y_axis.majorUnit  = pm_unit
            c_pm.set_categories(cats)
            c_pm.series[0].graphicalProperties.line.solidFill = "C0504D"  # red

            c_uv += c_pm

            ws.add_chart(c_uv, f"{anchor_col}{anchor_row}")
            anchor_row += 26

        col_cursor += len(headers) + 1
        
def build_correlations_sheet(wb, seg_refs):
    ws = wb.create_sheet("Correlations")
    ws.sheet_view.showGridLines = False
    ws.cell(row=1, column=1, value="Correlation Matrices (all sensors, per run)").font = TITLE_FONT

    explain = [
        "What is a correlation matrix?",
        "A correlation matrix shows how every pair of sensor readings moves together, on a scale from -1 to 1.",
        "  +1.00 = the two readings rise and fall together (strong positive relationship)",
        "  -1.00 = one rises exactly as the other falls (strong negative relationship)",
        "   0.00 = no linear relationship between the two readings",
        "The diagonal is always 1.00 (every sensor perfectly correlates with itself), and the matrix is symmetric.",
        "\"n/a\" appears when a sensor is constant for that run (e.g. UV_Raw = 0 throughout), making correlation undefined.",
        "All values are live Excel CORREL() formulas - they recalculate automatically if the source data changes.",
    ]
    for i, line in enumerate(explain, start=3):
        ws.merge_cells(start_row=i, start_column=1, end_row=i, end_column=10)
        c = ws.cell(row=i, column=1, value=line)
        c.font = SUB_FONT if i == 3 else BODY_FONT
    row_cursor = 3 + len(explain) + 2

    metrics = METRICS  # all sensor columns, excludes Time_Stamp_s
    n = len(metrics)

    for seg_label, (sheetname, r1, r2) in seg_refs.items():
        ws.cell(row=row_cursor, column=1, value=f"{seg_label} - Correlation Matrix").font = SUB_FONT
        row_cursor += 1
        header_row = row_cursor
        for j, m in enumerate(metrics, start=2):
            ws.cell(row=header_row, column=j, value=m)

        ranges = {m: f"{sheetname}!{col_letter_in(sheetname, m)}{r1}:{col_letter_in(sheetname, m)}{r2}" for m in metrics}

        for i, m_row in enumerate(metrics, start=header_row + 1):
            ws.cell(row=i, column=1, value=m_row).font = BODY_FONT
            for j, m_col in enumerate(metrics, start=2):
                if m_row == m_col:
                    ws.cell(row=i, column=j, value=1)
                else:
                    ws.cell(row=i, column=j, value=f'=IFERROR(CORREL({ranges[m_row]},{ranges[m_col]}),"n/a")')
                ws.cell(row=i, column=j).number_format = "0.00"

        style_header_row(ws, header_row, n + 1)
        row_cursor = header_row + n + 3   # gap before next run's matrix

    autosize(ws, n + 1, width=14)
    ws.column_dimensions['A'].width = 24


In [37]:
# Main
def main():
    print(f"Loading {SRC} ...")
    segments = load_data(SRC)
    for label, df in segments.items():
        print(f"  {label}: {len(df):,} rows, duration {df['Time_Stamp_s'].max()}s")
 
    seg_info = {label: describe_segment(label, df) for label, df in segments.items()}
 
    wb = Workbook()
    wb.remove(wb.active)
 
    build_overview_sheet(wb, seg_info)
    n_s1, seg_row_ranges = build_data_sheets(wb, segments)
    seg_refs = build_segment_stats_sheet(wb, segments, n_s1, seg_row_ranges)
    build_trends_sheet(wb, segments)
    build_correlations_sheet(wb, seg_refs)
 
    wb.save(OUT)
    print(f"Done -> {OUT}")
    print("\nNote: formulas (MIN/AVERAGE/MAX/STDEV/CORREL) are written as formula strings.")
    print("Open the file in Excel and it will recalculate them automatically on open.")
 
 
if __name__ == "__main__":
    main()


Loading HERA Data.xlsx ...
  S1: 12,541 rows, duration 12540s
  S2_A: 1,995 rows, duration 1994s
  S2_B: 14,426 rows, duration 14425s
  S2_C: 14,426 rows, duration 14425s
Done -> HERAAnalysis.xlsx

Note: formulas (MIN/AVERAGE/MAX/STDEV/CORREL) are written as formula strings.
Open the file in Excel and it will recalculate them automatically on open.
